# Decision Tree Classifier — UCI Adult Census Income Dataset

This notebook demonstrates decision tree classification on the Adult Census Income dataset. By the end you will be able to:

- Train a decision tree and evaluate it with a confusion matrix and classification report
- Observe overfitting and underfitting by sweeping `max_depth`
- Interpret feature importances to understand which features drive income predictions
- Visualise the decision boundary in 2D via PCA projection

## Mathematical Intuition

A decision tree partitions the feature space by recursively selecting the split that maximises **information gain**.

**Entropy** measures the impurity of a node with class distribution $\{p_k\}$:

$$H = -\sum_{k} p_k \log_2 p_k$$

A pure node (all one class) has $H = 0$; a maximally mixed node has $H = \log_2 K$ for $K$ classes.

**Information gain** for a split into children $L$ and $R$ is:

$$\text{IG} = H(\text{parent}) - \frac{n_L}{n} H(L) - \frac{n_R}{n} H(R)$$

The algorithm greedily selects the feature and threshold that maximise IG at each node. Splitting continues until `max_depth` is reached or no further gain is possible.

**Feature importance** is the total weighted IG contributed by a feature across all nodes where it is used as the split variable, normalised to sum to 1.

## Dataset Overview

The **UCI Adult Census Income** dataset contains demographic and employment information from the 1994 US Census. After removing rows with missing values, approximately 30,000 records remain.

| Feature | Type | Description |
|---------|------|-------------|
| age | Continuous | Age (years) |
| fnlwgt | Continuous | Final sampling weight |
| education-num | Continuous | Years of education (encoded) |
| capital-gain | Continuous | Capital gains income |
| capital-loss | Continuous | Capital losses |
| hours-per-week | Continuous | Hours worked per week |
| **target** | **Binary** | **1 = income >50K, 0 = income ≤50K** |

- **Rows:** ~30,000 (after dropna)  
- **Features used:** 6 (numeric only)  
- **Target:** binary income threshold

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlpackage import (
    DecisionTreeClassifier, PCA,
    train_test_split,
    confusion_matrix, classification_report,
)

sns.set_style("whitegrid")

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
df = pd.read_csv(url, names=cols, na_values=" ?", skipinitialspace=True)
df.dropna(inplace=True)
df["target"] = (df["income"] == ">50K").astype(int)

print("Shape after dropna:", df.shape)
df.head(3)

## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df["target"].value_counts().plot(kind="bar", color=["steelblue", "coral"], ax=ax)
ax.set_title("Target Distribution")
ax.set_xlabel("Income >50K (0 = No, 1 = Yes)")
ax.set_ylabel("Count")
ax.set_xticklabels(["<=50K", ">50K"], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
numeric_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), numeric_features):
    for val, color in [(0, "steelblue"), (1, "coral")]:
        ax.hist(df.loc[df["target"] == val, col], bins=30, alpha=0.6,
                color=color, label=str(val), edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend(title=">50K")
plt.suptitle("Feature Distributions by Income Class", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df[numeric_features + ["target"]].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
X = df[numeric_features].values.astype(float)
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test  class balance: {np.bincount(y_test)}")

## Model Training

In [ ]:
tree = DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=42)
tree.fit(X_train, y_train)

train_acc = tree.score(X_train, y_train)
test_acc  = tree.score(X_test,  y_test)
print(f"Train accuracy: {train_acc:.4f}  |  Test accuracy: {test_acc:.4f}")

In [ ]:
# Depth sweep
depths      = [1, 2, 3, 5, 8, 12, None]
depth_labels = [str(d) for d in depths]
train_accs   = []
test_accs    = []

for d in depths:
    t = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(t.score(X_train, y_train))
    test_accs.append(t.score(X_test,   y_test))
    print(f"max_depth={str(d):4s}  Train: {train_accs[-1]:.4f}  |  Test: {test_accs[-1]:.4f}")

best_idx   = int(np.argmax(test_accs))
best_depth = depths[best_idx]
print(f"\nBest test accuracy at max_depth={best_depth}: {test_accs[best_idx]:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depth_labels, train_accs, marker="o", label="Train accuracy", color="steelblue")
ax.plot(depth_labels, test_accs,  marker="o", label="Test accuracy",  color="coral")
ax.axvline(x=depth_labels[best_idx], color="gray", linestyle="--", label=f"Best depth={best_depth}")
ax.set_title("Accuracy vs max_depth — Decision Tree")
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

## Evaluation

In [ ]:
best_tree = DecisionTreeClassifier(criterion="entropy", max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)
y_pred = best_tree.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"])
ax.set_title(f"Confusion Matrix — Decision Tree (max_depth={best_depth})")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

classification_report(y_test, y_pred)

## Visualisations

In [ ]:
# Feature importance bar chart
importances = best_tree.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(np.array(numeric_features)[sorted_idx], importances[sorted_idx], color="steelblue")
ax.set_title("Feature Importances — Best Decision Tree")
ax.set_xlabel("Feature")
ax.set_ylabel("Importance (normalised information gain)")
plt.tight_layout()
plt.show()

In [ ]:
# Decision boundary via PCA 2D projection
from mlpackage import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d  = pca.transform(X_test_scaled)

tree_2d = DecisionTreeClassifier(criterion="entropy", max_depth=best_depth, random_state=42)
tree_2d.fit(X_train_2d, y_train)
print(f"2D tree  Train: {tree_2d.score(X_train_2d, y_train):.4f}  |  Test: {tree_2d.score(X_test_2d, y_test):.4f}")

h = 0.1
x_min, x_max = X_train_2d[:, 0].min() - 0.5, X_train_2d[:, 0].max() + 0.5
y_min, y_max = X_train_2d[:, 1].min() - 0.5, X_train_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = tree_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
for cls, color, marker in [(0, "steelblue", "o"), (1, "coral", "^")]:
    mask = y_test == cls
    ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
               c=color, marker=marker, s=10, alpha=0.5, label=f"Income >50K: {cls}")
ax.set_title(f"Decision Boundary — Decision Tree max_depth={best_depth} (PCA 2D)")
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation and Conclusions

- **The depth sweep reveals classic overfitting behaviour.** Shallow trees underfit (low train and test accuracy), while deep trees overfit (high train accuracy, decreasing test accuracy as depth grows beyond the optimum). The dashed line marks the sweet spot.

- **capital-gain is the dominant feature** according to the importance scores. This makes intuitive sense: large capital gains are rare but strongly associated with high income, giving this feature a high information gain at the root.

- **education-num and age also contribute substantially**, reflecting the well-established relationship between educational attainment, age, and income.

- **The confusion matrix shows the class imbalance effect:** because most people earn ≤50K, the tree achieves good overall accuracy but has lower recall for the minority class (>50K). Precision and recall for the positive class are the more informative metrics here.

- **The PCA decision boundary is piecewise rectangular**, which is the hallmark of axis-aligned splits produced by decision trees. Unlike logistic regression, the boundary need not be a single hyperplane.